# Volcano plots: differential gene and transcript isoform expression

One objective: show which genes and which transcript isoforms differ between the two groups, from the edgeR tables the pipeline wrote in S4.

In [ ]:
source("src/volcano.R")
getwd()


**Explanation.** The first line loads our plotting function. The line that starts with `loc` reads `paths.json`, the one place where the run's results folder, the comparison name and the dataset name are set. The other cells use short names such as `loc$de_gene`, so to look at another run you change `paths.json` and nothing in the notebook.

In [ ]:
source("src/paths.R")
options(repr.plot.width = 7, repr.plot.height = 5)
loc <- run_locations()        # every folder of the run, from paths.json
comparison <- loc$comparison

In [ ]:
cls_file <- Sys.glob(file.path(loc$sqanti_protein, "*best_ORF_SQANTI_classification.tsv"))[1]
cls <- fread(cls_file, select = c("isoform_id", "tx_cat", "pr_splice_cat"), data.table = FALSE)
# "known" = SQANTI full-splice_match, "novel" = any other category, "no ORF" = isoform without a predicted ORF
novelty_of <- function(ids, cat) {
  i <- match(ids, cls$isoform_id)
  ifelse(is.na(i), "no ORF", ifelse(cat[i] == "full-splice_match", "known", "novel"))
}

**Explanation.** The SQANTI classification file lists, for every isoform that has a predicted ORF, how its splicing compares with the reference annotation (GENCODE). `full-splice_match` means the isoform has the same splice junctions as a known transcript; we call that **known**. Any other category (new junctions, new combinations of known junctions, an incomplete copy of a known transcript, and so on) is called **novel**. An isoform with no predicted ORF is not in the file, so it is called **no ORF**. For the isoform plot we use the category of the transcript (`tx_cat`); for the ORF plot we use the category of the predicted protein (`pr_splice_cat`).

In [ ]:
dge <- read_edger_results(file.path(loc$de_gene, paste0(comparison, "_DGE_edgeR_results.txt")))
save_plot(volcano_plot(dge, label_col = "gene_name", title = "Genes"), "volcano_genes")

**Explanation.** Each point is a gene. Left to right is how much more or less it is expressed in one group (log2 fold change); height is how sure we are (-log10 of the false discovery rate, FDR). Dashed lines mark FDR 0.05 and a two-fold change; coloured points pass both. The most significant ones are named. Which group is 'up' is the group edgeR used as its second group, so check the M2 log before putting words on the figure.

In [ ]:
dte <- read_edger_results(file.path(loc$de_transcript, paste0(comparison, "_DTE_edgeR_results.txt")))
dte_novelty <- novelty_of(dte$isoform_id, cls$tx_cat)
table(changed = dte$FDR < 0.05 & abs(dte$logFC) >= 1, novelty = dte_novelty)   # how many of each class pass the cut-offs
save_plot(volcano_plot(dte, label_col = "transcript_name", title = "Transcript isoforms", novelty = dte_novelty), "volcano_isoforms")

**Explanation.** The same plot for transcript isoforms. A gene can have one isoform that goes up and another that does not change, which the gene plot cannot show. Changed points are coloured by whether the isoform is known or novel (see above), and the legend gives how many of each class there are, so the isoforms the long reads discovered stand out. Grey points do not pass the cut-offs. The table above the plot counts the same classes for the points that pass (TRUE) and those that do not (FALSE). Novel isoforms have no name, so their label is their id.

In [ ]:
orf_file <- file.path(loc$de_orf, paste0(comparison, "_DE_ORF_edgeR_results.txt"))
if (file.exists(orf_file)) {
  orf <- read_edger_results(orf_file)
  orf_novelty <- novelty_of(orf$orf_base_id, cls$pr_splice_cat)
  print(table(changed = orf$FDR < 0.05 & abs(orf$logFC) >= 1, novelty = orf_novelty))
  save_plot(volcano_plot(orf, label_col = "orf_base_id", title = "ORFs", novelty = orf_novelty), "volcano_orfs")
} else {
  message("Not found: ", orf_file)
}

**Explanation.** The same plot for open reading frames (ORFs), the predicted protein-coding stretches; each point is one ORF, named by its first transcript. If the results file is not in the results folder, the cell says so instead of failing. Colours as for the isoforms, using the class of the predicted protein.